# Stencil on the GPU (chapter 8)

Runs every program of `ec2/stencil` on a Colab GPU. First choose **Runtime → Change runtime type → GPU (T4 is enough)**, then **Runtime → Run all**.

The theory is in `ec2/stencil/README.md`. The same `.cu` files also run on a laptop without a GPU: `clang++ -std=c++17 -O2 -pthread -x c++ -I../common FILE.cu -o FILE && ./FILE`.

In [ ]:
!nvidia-smi
!nvcc --version | tail -2

The compatibility header: plain CUDA under `nvcc`, a CPU emulation under `g++`/`clang++`.

In [ ]:
%%writefile cuda_compat.h
#pragma once

#include <chrono>
#include <cstdio>
#include <cstdlib>

#ifdef __CUDACC__

#include <cuda_runtime.h>
#include <cuda/atomic>

#define CUDA_BACKEND "GPU (nvcc)"
#define BY_BACKEND(gpu, cpu) (gpu)

template <class Kernel, class... Args>
void launch(Kernel kernel, dim3 grid, dim3 block, Args... args) {
  kernel<<<grid, block>>>(args...);
}

#else

#include <atomic>
#include <cstring>
#include <memory>
#include <thread>
#include <type_traits>
#include <vector>

#define CUDA_BACKEND "CPU emulation (threads)"
#define BY_BACKEND(gpu, cpu) (cpu)

#define __global__
#define __device__
#define __host__
#define __constant__
#define __shared__ static
#define __restrict__

struct dim3 {
  unsigned x;
  unsigned y;
  unsigned z;

  dim3(unsigned x_ = 1, unsigned y_ = 1, unsigned z_ = 1) : x(x_), y(y_), z(z_) {
  }
};

struct uint3 {
  unsigned x;
  unsigned y;
  unsigned z;
};

struct float4 {
  float x;
  float y;
  float z;
  float w;
};

inline thread_local uint3 threadIdx{0, 0, 0};
inline thread_local uint3 blockIdx{0, 0, 0};
inline dim3 blockDim;
inline dim3 gridDim;

constexpr int warpSize = 32;

namespace emu {

constexpr unsigned kWarpSize = warpSize;
constexpr unsigned kFullWarpMask = 0xffffffffu;
constexpr int kCudaAtomicFunctionOrder = __ATOMIC_RELAXED;

class Barrier {
 public:
  explicit Barrier(unsigned threadCount) : threadCount_(threadCount) {
  }

  void arriveAndWait() {
    const unsigned generationOnArrival = generation_.load(std::memory_order_acquire);
    const bool lastToArrive = arrived_.fetch_add(1, std::memory_order_acq_rel) + 1 == threadCount_;

    if (lastToArrive) {
      releaseEveryWaitingThread();
    } else {
      yieldUntilGenerationEnds(generationOnArrival);
    }
  }

 private:
  void releaseEveryWaitingThread() {
    arrived_.store(0, std::memory_order_relaxed);
    generation_.fetch_add(1, std::memory_order_acq_rel);
  }

  void yieldUntilGenerationEnds(unsigned generation) const {
    while (generation_.load(std::memory_order_acquire) == generation) {
      std::this_thread::yield();
    }
  }

  std::atomic<unsigned> arrived_{0};
  std::atomic<unsigned> generation_{0};
  const unsigned threadCount_;
};

inline Barrier* blockBarrier = nullptr;

using LaneBits = unsigned long long;

class Warp {
 public:
  explicit Warp(unsigned laneCount) : laneCount_(laneCount), barrier_(laneCount), published_{} {
  }

  unsigned laneCount() const {
    return laneCount_;
  }

  unsigned everyLaneMask() const {
    if (laneCount_ == kWarpSize) {
      return kFullWarpMask;
    }
    return (1u << laneCount_) - 1u;
  }

  bool hasLane(int lane) const {
    return lane >= 0 && lane < static_cast<int>(laneCount_);
  }

  void publish(unsigned lane, LaneBits bits) {
    __atomic_store_n(&published_[lane], bits, __ATOMIC_RELAXED);
  }

  LaneBits publishedBy(int lane) const {
    return __atomic_load_n(&published_[lane], __ATOMIC_RELAXED);
  }

  void waitForEveryLane() {
    barrier_.arriveAndWait();
  }

 private:
  const unsigned laneCount_;
  Barrier barrier_;
  LaneBits published_[kWarpSize];
};

inline thread_local Warp* warpOfThisThread = nullptr;
inline thread_local unsigned laneOfThisThread = 0;

template <class T>
LaneBits toLaneBits(T value) {
  static_assert(sizeof(T) <= sizeof(LaneBits), "shuffle of a type wider than 8 bytes");
  LaneBits bits = 0;
  std::memcpy(&bits, &value, sizeof(T));
  return bits;
}

template <class T>
T fromLaneBits(LaneBits bits) {
  T value;
  std::memcpy(&value, &bits, sizeof(T));
  return value;
}

template <class T>
T readValueOfLane(T myValue, int sourceLane) {
  Warp& myWarp = *warpOfThisThread;
  myWarp.publish(laneOfThisThread, toLaneBits(myValue));
  myWarp.waitForEveryLane();

  T received = myValue;
  if (myWarp.hasLane(sourceLane)) {
    received = fromLaneBits<T>(myWarp.publishedBy(sourceLane));
  }

  myWarp.waitForEveryLane();
  return received;
}

inline int positionInGroup(int lane, int width) {
  return lane % width;
}

inline int firstLaneOfGroup(int lane, int width) {
  return lane / width * width;
}

inline bool inSameGroup(int lane, int otherLane, int width) {
  return lane / width == otherLane / width;
}

inline int shflDownSourceLane(int lane, int delta, int width) {
  const bool sourceInGroup = positionInGroup(lane, width) + delta < width;
  return sourceInGroup ? lane + delta : lane;
}

inline int shflUpSourceLane(int lane, int delta, int width) {
  const bool sourceInGroup = positionInGroup(lane, width) >= delta;
  return sourceInGroup ? lane - delta : lane;
}

inline int shflXorSourceLane(int lane, int laneMask, int width) {
  const int partner = lane ^ laneMask;
  return inSameGroup(lane, partner, width) ? partner : lane;
}

inline int shflSourceLane(int lane, int srcLane, int width) {
  const int wrappedSrcLane = (srcLane % width + width) % width;
  return firstLaneOfGroup(lane, width) + wrappedSrcLane;
}

inline int myLane() {
  return static_cast<int>(laneOfThisThread);
}

}

inline void __syncthreads() {
  emu::blockBarrier->arriveAndWait();
}

inline void __syncwarp([[maybe_unused]] unsigned mask = emu::kFullWarpMask) {
  emu::warpOfThisThread->waitForEveryLane();
}

template <class T>
T __shfl_down_sync([[maybe_unused]] unsigned mask, T var, unsigned delta, int width = warpSize) {
  const int sourceLane = emu::shflDownSourceLane(emu::myLane(), static_cast<int>(delta), width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_up_sync([[maybe_unused]] unsigned mask, T var, unsigned delta, int width = warpSize) {
  const int sourceLane = emu::shflUpSourceLane(emu::myLane(), static_cast<int>(delta), width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_xor_sync([[maybe_unused]] unsigned mask, T var, int laneMask, int width = warpSize) {
  const int sourceLane = emu::shflXorSourceLane(emu::myLane(), laneMask, width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_sync([[maybe_unused]] unsigned mask, T var, int srcLane, int width = warpSize) {
  const int sourceLane = emu::shflSourceLane(emu::myLane(), srcLane, width);
  return emu::readValueOfLane(var, sourceLane);
}

inline unsigned __ballot_sync([[maybe_unused]] unsigned mask, int predicate) {
  unsigned ballot = 0;
  const unsigned laneCount = emu::warpOfThisThread->laneCount();

  for (unsigned sourceLane = 0; sourceLane < laneCount; ++sourceLane) {
    const int predicateOfLane =
        __shfl_sync(emu::kFullWarpMask, predicate, static_cast<int>(sourceLane));
    if (predicateOfLane) {
      ballot |= 1u << sourceLane;
    }
  }

  return ballot;
}

inline int __any_sync(unsigned mask, int predicate) {
  return __ballot_sync(mask, predicate) != 0;
}

inline int __all_sync(unsigned mask, int predicate) {
  return __ballot_sync(mask, predicate) == emu::warpOfThisThread->everyLaneMask();
}

namespace emu {

using WarpList = std::vector<std::unique_ptr<Warp>>;

inline unsigned threadCountOf(dim3 block) {
  return block.x * block.y * block.z;
}

inline WarpList splitIntoWarps(unsigned threadCount) {
  WarpList warps;

  for (unsigned firstThread = 0; firstThread < threadCount; firstThread += kWarpSize) {
    const unsigned threadsLeft = threadCount - firstThread;
    const unsigned laneCount = threadsLeft < kWarpSize ? threadsLeft : kWarpSize;
    warps.push_back(std::make_unique<Warp>(laneCount));
  }

  return warps;
}

inline uint3 threadIndexOf(unsigned linearThread, dim3 block) {
  const unsigned x = linearThread % block.x;
  const unsigned y = linearThread / block.x % block.y;
  const unsigned z = linearThread / (block.x * block.y);
  return {x, y, z};
}

inline void waitForTheWholeBlockToFinish() {
  blockBarrier->arriveAndWait();
}

template <class Kernel, class... Args>
void runThreadInEveryBlock(
    unsigned linearThread, Warp* threadWarp, dim3 grid, dim3 block, Kernel kernel, Args... args) {
  warpOfThisThread = threadWarp;
  laneOfThisThread = linearThread % kWarpSize;
  threadIdx = threadIndexOf(linearThread, block);

  for (unsigned bz = 0; bz < grid.z; ++bz) {
    for (unsigned by = 0; by < grid.y; ++by) {
      for (unsigned bx = 0; bx < grid.x; ++bx) {
        blockIdx = {bx, by, bz};
        kernel(args...);
        waitForTheWholeBlockToFinish();
      }
    }
  }
}

}

template <class Kernel, class... Args>
void launch(Kernel kernel, dim3 grid, dim3 block, Args... args) {
  gridDim = grid;
  blockDim = block;

  const unsigned threadCount = emu::threadCountOf(block);
  emu::Barrier barrier(threadCount);
  emu::blockBarrier = &barrier;
  emu::WarpList warps = emu::splitIntoWarps(threadCount);

  std::vector<std::thread> threads;
  threads.reserve(threadCount);
  for (unsigned linearThread = 0; linearThread < threadCount; ++linearThread) {
    emu::Warp* threadWarp = warps[linearThread / emu::kWarpSize].get();
    threads.emplace_back([=] {
      emu::runThreadInEveryBlock(linearThread, threadWarp, grid, block, kernel, args...);
    });
  }

  for (std::thread& thread : threads) {
    thread.join();
  }
  emu::blockBarrier = nullptr;
}

namespace emu {

template <class T, class Improves>
T atomicReplaceWhileImproving(T* address, T candidate, Improves improves) {
  T current = __atomic_load_n(address, kCudaAtomicFunctionOrder);

  while (improves(candidate, current)) {
    const bool replaced = __atomic_compare_exchange_n(
        address, &current, candidate, true, kCudaAtomicFunctionOrder, kCudaAtomicFunctionOrder);
    if (replaced) {
      break;
    }
  }

  return current;
}

template <class T>
T atomicStoreMinimum(T* address, T value) {
  return atomicReplaceWhileImproving(address, value, [](T candidate, T current) {
    return candidate < current;
  });
}

template <class T>
T atomicStoreMaximum(T* address, T value) {
  return atomicReplaceWhileImproving(address, value, [](T candidate, T current) {
    return candidate > current;
  });
}

template <class T>
T atomicCompareAndSwap(T* address, T compare, T value) {
  T observed = compare;
  __atomic_compare_exchange_n(
      address, &observed, value, false, kCudaAtomicFunctionOrder, kCudaAtomicFunctionOrder);
  return observed;
}

}

inline int atomicAdd(int* address, int value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline unsigned atomicAdd(unsigned* address, unsigned value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline unsigned long long atomicAdd(unsigned long long* address, unsigned long long value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline float atomicAdd(float* address, float value) {
  float old;
  __atomic_load(address, &old, emu::kCudaAtomicFunctionOrder);
  float sum = old + value;

  while (!__atomic_compare_exchange(
      address, &old, &sum, true, emu::kCudaAtomicFunctionOrder, emu::kCudaAtomicFunctionOrder)) {
    sum = old + value;
  }

  return old;
}

inline int atomicMin(int* address, int value) {
  return emu::atomicStoreMinimum(address, value);
}

inline int atomicMax(int* address, int value) {
  return emu::atomicStoreMaximum(address, value);
}

inline unsigned long long atomicMin(unsigned long long* address, unsigned long long value) {
  return emu::atomicStoreMinimum(address, value);
}

inline unsigned atomicMin(unsigned* address, unsigned value) {
  return emu::atomicStoreMinimum(address, value);
}

inline unsigned atomicMax(unsigned* address, unsigned value) {
  return emu::atomicStoreMaximum(address, value);
}

inline int atomicSub(int* address, int value) {
  return __atomic_fetch_sub(address, value, emu::kCudaAtomicFunctionOrder);
}

inline int atomicExch(int* address, int value) {
  return __atomic_exchange_n(address, value, emu::kCudaAtomicFunctionOrder);
}

inline int atomicCAS(int* address, int compare, int value) {
  return emu::atomicCompareAndSwap(address, compare, value);
}

inline unsigned long long atomicCAS(unsigned long long* address,
                                    unsigned long long compare,
                                    unsigned long long value) {
  return emu::atomicCompareAndSwap(address, compare, value);
}

inline void __threadfence() {
  __atomic_thread_fence(__ATOMIC_SEQ_CST);
}

namespace cuda {

enum thread_scope {
  thread_scope_system = 0,
  thread_scope_device = 1,
  thread_scope_block = 2,
  thread_scope_thread = 3
};

namespace std {

enum memory_order {
  memory_order_relaxed = __ATOMIC_RELAXED,
  memory_order_consume = __ATOMIC_CONSUME,
  memory_order_acquire = __ATOMIC_ACQUIRE,
  memory_order_release = __ATOMIC_RELEASE,
  memory_order_acq_rel = __ATOMIC_ACQ_REL,
  memory_order_seq_cst = __ATOMIC_SEQ_CST
};

}

using std::memory_order;
using std::memory_order_acq_rel;
using std::memory_order_acquire;
using std::memory_order_relaxed;
using std::memory_order_release;
using std::memory_order_seq_cst;

template <class T, thread_scope Scope = thread_scope_system>
class atomic_ref {
 public:
  explicit atomic_ref(T& object) : address_(&object) {
  }

  T load(memory_order order = memory_order_seq_cst) const {
    T value;
    __atomic_load(address_, &value, order);
    return value;
  }

  void store(T value, memory_order order = memory_order_seq_cst) const {
    __atomic_store(address_, &value, order);
  }

  T exchange(T value, memory_order order = memory_order_seq_cst) const {
    T old;
    __atomic_exchange(address_, &value, &old, order);
    return old;
  }

  bool compare_exchange_strong(T& expected,
                               T desired,
                               memory_order order = memory_order_seq_cst) const {
    return __atomic_compare_exchange(address_, &expected, &desired, false, order, __ATOMIC_RELAXED);
  }

  T fetch_add(T value, memory_order order = memory_order_seq_cst) const {
    if constexpr (::std::is_integral_v<T>) {
      return __atomic_fetch_add(address_, value, order);
    } else {
      return fetchAddByCompareExchange(value, order);
    }
  }

  T fetch_sub(T value, memory_order order = memory_order_seq_cst) const {
    return fetch_add(-value, order);
  }

  T fetch_min(T value, memory_order order = memory_order_seq_cst) const {
    return replaceWhileImproving(value, order, [](T candidate, T current) {
      return candidate < current;
    });
  }

  T fetch_max(T value, memory_order order = memory_order_seq_cst) const {
    return replaceWhileImproving(value, order, [](T candidate, T current) {
      return candidate > current;
    });
  }

 private:
  T fetchAddByCompareExchange(T value, memory_order order) const {
    T old = load(memory_order_relaxed);
    bool replaced = false;
    while (!replaced) {
      replaced = compare_exchange_strong(old, old + value, order);
    }
    return old;
  }

  template <class Improves>
  T replaceWhileImproving(T candidate, memory_order order, Improves improves) const {
    T current = load(memory_order_relaxed);
    while (improves(candidate, current)) {
      if (compare_exchange_strong(current, candidate, order)) {
        break;
      }
    }
    return current;
  }

  T* address_;
};

}

enum cudaMemcpyKind {
  cudaMemcpyHostToDevice,
  cudaMemcpyDeviceToHost,
  cudaMemcpyDeviceToDevice,
  cudaMemcpyHostToHost
};

typedef int cudaError_t;

const cudaError_t cudaSuccess = 0;

inline const char* cudaGetErrorString(cudaError_t) {
  return "no error";
}

inline cudaError_t cudaGetLastError() {
  return cudaSuccess;
}

inline cudaError_t cudaDeviceSynchronize() {
  return cudaSuccess;
}

template <class T>
cudaError_t cudaMalloc(T** devicePointer, size_t bytes) {
  *devicePointer = static_cast<T*>(std::malloc(bytes));
  return cudaSuccess;
}

inline cudaError_t cudaFree(void* devicePointer) {
  std::free(devicePointer);
  return cudaSuccess;
}

inline cudaError_t cudaMemcpy(void* destination, const void* source, size_t bytes, cudaMemcpyKind) {
  std::memcpy(destination, source, bytes);
  return cudaSuccess;
}

inline cudaError_t cudaMemset(void* destination, int byteValue, size_t bytes) {
  std::memset(destination, byteValue, bytes);
  return cudaSuccess;
}

template <class T>
cudaError_t cudaMemcpyToSymbol(T& symbol, const void* source, size_t bytes) {
  std::memcpy(&symbol, source, bytes);
  return cudaSuccess;
}

#endif

#define CHECK(call)                                                                          \
  do {                                                                                       \
    cudaError_t err_ = (call);                                                               \
    if (err_ != cudaSuccess) {                                                               \
      std::fprintf(                                                                          \
          stderr, "CUDA error %s at %s:%d\n", cudaGetErrorString(err_), __FILE__, __LINE__); \
      std::exit(1);                                                                          \
    }                                                                                        \
  } while (0)

struct Timer {
  std::chrono::steady_clock::time_point t0 = std::chrono::steady_clock::now();

  double ms() {
    cudaDeviceSynchronize();
    const auto elapsed = std::chrono::steady_clock::now() - t0;
    return std::chrono::duration<double, std::milli>(elapsed).count();
  }
};

inline unsigned cdiv(unsigned a, unsigned b) {
  return (a + b - 1) / b;
}

__host__ __device__ inline int imin(int a, int b) {
  return a < b ? a : b;
}

__host__ __device__ inline int imax(int a, int b) {
  return a > b ? a : b;
}

__host__ __device__ inline float fminf_(float a, float b) {
  return a < b ? a : b;
}

__host__ __device__ inline float fmaxf_(float a, float b) {
  return a > b ? a : b;
}


## `bikeshare_station_coverage.cu`

In [ ]:
%%writefile bikeshare_station_coverage.cu
#include "cuda_compat.h"

#include <algorithm>
#include <cmath>
#include <cstdint>
#include <cstdio>
#include <string>
#include <utility>
#include <vector>

#ifndef CPU_GRID_SIDE
#define CPU_GRID_SIDE 40
#endif

const int GRID_SIDE = BY_BACKEND(160, CPU_GRID_SIDE);
const int MAP_SIDE_METRES = 6000;

#define IN_TILE 16
#define OUT_TILE (IN_TILE - 2)

__global__ void jacobi5_tiled_kernel(const float* in,
                                     float* out,
                                     const unsigned char* fixed,
                                     float cC,
                                     float cN,
                                     int width,
                                     int height) {
  int col = (int)(blockIdx.x * OUT_TILE) + (int)threadIdx.x - 1;
  int row = (int)(blockIdx.y * OUT_TILE) + (int)threadIdx.y - 1;
  int tx = (int)threadIdx.x;
  int ty = (int)threadIdx.y;

  __shared__ float u_s[IN_TILE][IN_TILE];
  bool inside = row >= 0 && row < height && col >= 0 && col < width;
  u_s[ty][tx] = inside ? in[row * width + col] : 0.0f;
  __syncthreads();

  if (inside && tx >= 1 && tx < IN_TILE - 1 && ty >= 1 && ty < IN_TILE - 1) {
    int p = row * width + col;
    if (fixed[p]) {
      out[p] = u_s[ty][tx];
    } else {
      out[p] = cC * u_s[ty][tx] +
               cN * (u_s[ty][tx - 1] + u_s[ty][tx + 1] + u_s[ty - 1][tx] + u_s[ty + 1][tx]);
    }
  }
}

__global__ void inject_kernel(float* u, const int* cells, int n) {
  int t = (int)(blockIdx.x * blockDim.x + threadIdx.x);
  if (t < n) {
    atomicAdd(&u[cells[t]], 1.0f);
  }
}

using Field = std::vector<float>;
using FixedMask = std::vector<unsigned char>;

static void jacobi5_cpu(
    const Field& in, Field& out, const FixedMask& fixed, float cC, float cN, int w, int h) {
  for (int r = 0; r < h; ++r) {
    for (int c = 0; c < w; ++c) {
      int p = r * w + c;
      if (fixed[p]) {
        out[p] = in[p];
        continue;
      }
      float left = c > 0 ? in[p - 1] : 0.0f;
      float right = c < w - 1 ? in[p + 1] : 0.0f;
      float up = r > 0 ? in[p - w] : 0.0f;
      float down = r < h - 1 ? in[p + w] : 0.0f;
      out[p] = cC * in[p] + cN * (left + right + up + down);
    }
  }
}

static double maxRelativeDifference(const Field& expected, const Field& actual) {
  double worst = 0.0;
  for (size_t i = 0; i < expected.size(); ++i) {
    double difference = std::fabs((double)expected[i] - actual[i]);
    worst = std::fmax(worst, difference / (1.0 + std::fabs((double)expected[i])));
  }
  return worst;
}

class SplitMix64 {
 public:
  explicit SplitMix64(uint64_t seed) : state_(seed) {
  }

  uint64_t next() {
    state_ += GOLDEN_RATIO_INCREMENT;
    uint64_t z = state_;
    z = (z ^ (z >> 30)) * FIRST_MIX_MULTIPLIER;
    z = (z ^ (z >> 27)) * SECOND_MIX_MULTIPLIER;
    return z ^ (z >> 31);
  }

  int below(int n) {
    return static_cast<int>(next() % static_cast<uint64_t>(n));
  }

  double uniform() {
    return static_cast<double>(next() >> UNIFORM_SHIFT) * UNIFORM_SCALE;
  }

  double standardNormal() {
    const double awayFromZero = 1.0 - uniform();
    const double angle = TWO_PI * uniform();
    return std::sqrt(-2.0 * std::log(awayFromZero)) * std::cos(angle);
  }

 private:
  static constexpr uint64_t GOLDEN_RATIO_INCREMENT = 0x9E3779B97F4A7C15ull;
  static constexpr uint64_t FIRST_MIX_MULTIPLIER = 0xBF58476D1CE4E5B9ull;
  static constexpr uint64_t SECOND_MIX_MULTIPLIER = 0x94D049BB133111EBull;
  static constexpr int UNIFORM_SHIFT = 11;
  static constexpr double UNIFORM_SCALE = 1.0 / 9007199254740992.0;
  static constexpr double TWO_PI = 6.283185307179586;

  uint64_t state_;
};

struct Position {
  int x;
  int y;
};

struct Station {
  int id;
  Position position;
};

const int STATION_COUNT = 48;
const uint64_t STATION_SEED = 48;
const int STATION_MARGIN_METRES = 300;

const int NEIGHBOURS_PER_CELL = 4;
const int ZONES_PER_SIDE = 4;
const int ZONE_SIDE_METRES = MAP_SIDE_METRES / ZONES_PER_SIDE;
const int MAP_COLUMNS = 40;
const int MAP_ROWS = 20;
const char* const MAP_SHADES = " .:-=+*#%@";
const int DARKEST_SHADE = 9;
const float SHADE_SCALE = 9.99f;
const char STATION_MARK = 'S';
const double MAX_RELATIVE_ERROR = 1e-4;

const double DECAY_LENGTH_METRES = 250.0;
const float COVERAGE_AT_STATION = 1.0f;
const double CONVERGENCE_TOLERANCE = 1e-5;
const int SWEEPS_BETWEEN_CONVERGENCE_CHECKS = BY_BACKEND(200, 25);
const int MAX_SWEEPS = 200000;
const float COVERAGE_MAP_SHADE_MAX = 0.5f;

const int FIRST_SIMULATED_MINUTE_OF_DAY = 17 * 60 + 30;
const int SIMULATED_MINUTES = 30;
const int SNAPSHOT_EVERY_MINUTES = 10;
const int MINUTES_PER_HOUR = 60;
const int BASE_GRID_SIDE = 40;
const int STEPS_PER_MINUTE_ON_BASE_GRID = 3;
constexpr float DIFFUSION_ALPHA = 0.20f;
constexpr float MAX_STABLE_DIFFUSION_ALPHA = 0.25f;
const double DEMAND_KEPT_PER_MINUTE = 0.913;
const uint64_t DEMAND_SEED = 19;
const int MIN_REQUESTS_PER_MINUTE = 6;
const int EXTRA_REQUESTS_RANGE = 8;
constexpr int INJECT_BLOCK_DIM = 64;
const int REQUESTS_TO_SHOW = 2;
const int SECONDS_BETWEEN_REQUESTS_OF_A_MINUTE = 1;

static_assert(DIFFUSION_ALPHA <= MAX_STABLE_DIFFUSION_ALPHA,
              "explicit diffusion steps are stable only for alpha <= 1/4");
static_assert(GRID_SIDE >= BASE_GRID_SIDE && GRID_SIDE % BASE_GRID_SIDE == 0,
              "the grid refines the base grid by a whole factor");
static_assert(MIN_REQUESTS_PER_MINUTE + EXTRA_REQUESTS_RANGE - 1 <= INJECT_BLOCK_DIM,
              "one block of inject_kernel covers the busiest simulated minute");

enum class StationMarks { Show, Hide };

struct Zone {
  int row;
  int col;
};

struct ZoneMeans {
  double mean[ZONES_PER_SIDE][ZONES_PER_SIDE];

  double at(Zone zone) const {
    return mean[zone.row][zone.col];
  }
};

struct StencilCoefficients {
  float cC;
  float cN;
};

static size_t cellCount() {
  return static_cast<size_t>(GRID_SIDE) * GRID_SIDE;
}

static int cellCol(int x) {
  return imin(GRID_SIDE - 1, x * GRID_SIDE / MAP_SIDE_METRES);
}

static int cellRow(int y) {
  return imin(GRID_SIDE - 1, (MAP_SIDE_METRES - 1 - y) * GRID_SIDE / MAP_SIDE_METRES);
}

static int cellOf(Position position) {
  return cellRow(position.y) * GRID_SIDE + cellCol(position.x);
}

static double cellSizeMetres() {
  return static_cast<double>(MAP_SIDE_METRES) / GRID_SIDE;
}

static bool onMapEdge(int row, int col) {
  return row == 0 || col == 0 || row == GRID_SIDE - 1 || col == GRID_SIDE - 1;
}

static bool insideMap(Position position) {
  return position.x >= 0 && position.x < MAP_SIDE_METRES && position.y >= 0 &&
         position.y < MAP_SIDE_METRES;
}

static Position randomPosition(SplitMix64& rng, int marginMetres) {
  const int span = MAP_SIDE_METRES - 2 * marginMetres;
  Position position;
  position.x = marginMetres + rng.below(span);
  position.y = marginMetres + rng.below(span);
  return position;
}

static std::vector<Station> makeStations() {
  SplitMix64 rng(STATION_SEED);
  std::vector<Station> stations(STATION_COUNT);

  for (int id = 0; id < STATION_COUNT; ++id) {
    stations[id].id = id;
    stations[id].position = randomPosition(rng, STATION_MARGIN_METRES);
  }

  return stations;
}

static std::string nameOf(Zone zone) {
  std::string name;
  name += (char)('A' + zone.row);
  name += (char)('1' + zone.col);
  return name;
}

static ZoneMeans zoneMeans(const Field& u) {
  double sum[ZONES_PER_SIDE][ZONES_PER_SIDE] = {};
  double count[ZONES_PER_SIDE][ZONES_PER_SIDE] = {};
  for (int r = 0; r < GRID_SIDE; ++r) {
    for (int c = 0; c < GRID_SIDE; ++c) {
      const int zoneRow = r * ZONES_PER_SIDE / GRID_SIDE;
      const int zoneCol = c * ZONES_PER_SIDE / GRID_SIDE;
      sum[zoneRow][zoneCol] += u[r * GRID_SIDE + c];
      count[zoneRow][zoneCol] += 1;
    }
  }

  ZoneMeans means;
  for (int i = 0; i < ZONES_PER_SIDE; ++i) {
    for (int j = 0; j < ZONES_PER_SIDE; ++j) {
      means.mean[i][j] = sum[i][j] / count[i][j];
    }
  }
  return means;
}

static bool isInnerZone(Zone zone) {
  const int firstInner = 1;
  const int lastInner = ZONES_PER_SIDE - 2;
  return zone.row >= firstInner && zone.row <= lastInner && zone.col >= firstInner &&
         zone.col <= lastInner;
}

static Position zoneCentre(Zone zone) {
  Position centre;
  centre.x = zone.col * ZONE_SIDE_METRES + ZONE_SIDE_METRES / 2;
  centre.y = MAP_SIDE_METRES - zone.row * ZONE_SIDE_METRES - ZONE_SIDE_METRES / 2;
  return centre;
}

static bool stationIsDrawnAt(const std::vector<Station>& stations, int mapRow, int mapCol) {
  for (const Station& station : stations) {
    if (cellRow(station.position.y) * MAP_ROWS / GRID_SIDE == mapRow &&
        cellCol(station.position.x) * MAP_COLUMNS / GRID_SIDE == mapCol) {
      return true;
    }
  }
  return false;
}

static char shadeOf(float value, float shadeMax) {
  return MAP_SHADES[imin(DARKEST_SHADE, (int)(SHADE_SCALE * value / shadeMax))];
}

static void printAsciiMap(const Field& u,
                          float shadeMax,
                          const std::vector<Station>& stations,
                          StationMarks marks) {
  for (int y = 0; y < MAP_ROWS; ++y) {
    std::printf("      |");
    for (int x = 0; x < MAP_COLUMNS; ++x) {
      const int r = (y * GRID_SIDE) / MAP_ROWS + GRID_SIDE / (2 * MAP_ROWS);
      const int c = (x * GRID_SIDE) / MAP_COLUMNS + GRID_SIDE / (2 * MAP_COLUMNS);
      char symbol = shadeOf(u[r * GRID_SIDE + c], shadeMax);
      if (marks == StationMarks::Show && stationIsDrawnAt(stations, y, x)) {
        symbol = STATION_MARK;
      }
      std::printf("%c", symbol);
    }
    std::printf("|\n");
  }
}

static FixedMask mapEdgeMask() {
  FixedMask fixed(cellCount(), 0);
  for (int i = 0; i < GRID_SIDE; ++i) {
    fixed[i] = 1;
    fixed[(GRID_SIDE - 1) * GRID_SIDE + i] = 1;
    fixed[i * GRID_SIDE] = 1;
    fixed[i * GRID_SIDE + GRID_SIDE - 1] = 1;
  }
  return fixed;
}

class DeviceField {
 public:
  DeviceField()
      : bytes_(cellCount() * sizeof(float)),
        block_(IN_TILE, IN_TILE),
        grid_(cdiv(GRID_SIDE, OUT_TILE), cdiv(GRID_SIDE, OUT_TILE)) {
    CHECK(cudaMalloc(&bufferA_, bytes_));
    CHECK(cudaMalloc(&bufferB_, bytes_));
    CHECK(cudaMalloc(&fixed_, cellCount()));
  }

  ~DeviceField() {
    CHECK(cudaFree(bufferA_));
    CHECK(cudaFree(bufferB_));
    CHECK(cudaFree(fixed_));
  }

  void upload(const Field& initial, const FixedMask& fixed) {
    CHECK(cudaMemcpy(bufferA_, initial.data(), bytes_, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(bufferB_, initial.data(), bytes_, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(fixed_, fixed.data(), cellCount(), cudaMemcpyHostToDevice));
    latest_ = bufferA_;
    older_ = bufferB_;
  }

  void sweep(StencilCoefficients coefficients) {
    launch(jacobi5_tiled_kernel,
           grid_,
           block_,
           (const float*)latest_,
           older_,
           (const unsigned char*)fixed_,
           coefficients.cC,
           coefficients.cN,
           GRID_SIDE,
           GRID_SIDE);
    std::swap(latest_, older_);
  }

  float* latest() {
    return latest_;
  }

  Field downloadLatest() const {
    return download(latest_);
  }

  Field downloadOneSweepBefore() const {
    return download(older_);
  }

 private:
  Field download(const float* buffer) const {
    Field field(cellCount());
    CHECK(cudaMemcpy(field.data(), buffer, bytes_, cudaMemcpyDeviceToHost));
    return field;
  }

  size_t bytes_;
  dim3 block_;
  dim3 grid_;
  float* bufferA_;
  float* bufferB_;
  unsigned char* fixed_;
  float* latest_ = nullptr;
  float* older_ = nullptr;
};

static void sweepOnCpu(Field& latest,
                       Field& older,
                       const FixedMask& fixed,
                       StencilCoefficients coefficients) {
  jacobi5_cpu(latest, older, fixed, coefficients.cC, coefficients.cN, GRID_SIDE, GRID_SIDE);
  std::swap(latest, older);
}

static double maxChange(const Field& latest, const Field& before) {
  double change = 0.0;
  for (size_t i = 0; i < latest.size(); ++i) {
    change = std::fmax(change, std::fabs(latest[i] - before[i]));
  }
  return change;
}

struct CoverageProblem {
  Field initial;
  FixedMask fixed;
};

static CoverageProblem coverageProblem(const std::vector<Station>& stations) {
  CoverageProblem problem;
  problem.initial.assign(cellCount(), 0.0f);
  problem.fixed = mapEdgeMask();

  for (const Station& station : stations) {
    const int cell = cellOf(station.position);
    problem.fixed[cell] = 1;
    problem.initial[cell] = COVERAGE_AT_STATION;
  }
  return problem;
}

static float screenedLaplaceLambda() {
  const double kappa = cellSizeMetres() / DECAY_LENGTH_METRES;
  return (float)(NEIGHBOURS_PER_CELL / (NEIGHBOURS_PER_CELL + kappa * kappa));
}

struct Convergence {
  Field coverage;
  int sweeps;
  double lastChange;
  double ms;
};

static Convergence sweepUntilConverged(DeviceField& field, StencilCoefficients coefficients) {
  Convergence result;
  result.sweeps = 0;
  result.lastChange = 1.0;

  Timer timer;
  while (result.lastChange > CONVERGENCE_TOLERANCE && result.sweeps < MAX_SWEEPS) {
    for (int s = 0; s < SWEEPS_BETWEEN_CONVERGENCE_CHECKS; ++s) {
      field.sweep(coefficients);
      ++result.sweeps;
    }
    CHECK(cudaGetLastError());
    result.coverage = field.downloadLatest();
    result.lastChange = maxChange(result.coverage, field.downloadOneSweepBefore());
  }
  result.ms = timer.ms();
  return result;
}

static Field coverageOnCpu(const CoverageProblem& problem,
                           StencilCoefficients coefficients,
                           int sweeps) {
  Field latest = problem.initial;
  Field older = problem.initial;
  for (int s = 0; s < sweeps; ++s) {
    sweepOnCpu(latest, older, problem.fixed, coefficients);
  }
  return latest;
}

static void printZoneMeans(const ZoneMeans& means) {
  std::printf("   mean coverage per zone (rows A..D north to south, columns 1..4 west to east):\n");
  for (int i = 0; i < ZONES_PER_SIDE; ++i) {
    std::printf("      %c ", 'A' + i);
    for (int j = 0; j < ZONES_PER_SIDE; ++j) {
      std::printf("  %6.4f", means.mean[i][j]);
    }
    std::printf("\n");
  }
}

static void printWorstZones(const ZoneMeans& means) {
  Zone worst = {0, 0};
  Zone worstInner = {1, 1};
  for (int i = 0; i < ZONES_PER_SIDE; ++i) {
    for (int j = 0; j < ZONES_PER_SIDE; ++j) {
      const Zone zone = {i, j};
      if (means.at(zone) < means.at(worst)) {
        worst = zone;
      }
      if (isInnerZone(zone) && means.at(zone) < means.at(worstInner)) {
        worstInner = zone;
      }
    }
  }

  const Position suggestion = zoneCentre(worstInner);
  std::printf("   worst covered zone: %s (%.4f); worst inner zone: %s (%.4f)\n",
              nameOf(worst).c_str(),
              means.at(worst),
              nameOf(worstInner).c_str(),
              means.at(worstInner));
  std::printf(
      "   suggestion: a new docking station near (%d m, %d m)\n", suggestion.x, suggestion.y);
}

static bool runStationCoverage(DeviceField& field, const std::vector<Station>& stations) {
  const float lambda = screenedLaplaceLambda();
  const float centreWeight = 0.0f;
  const float neighbourWeight = lambda / NEIGHBOURS_PER_CELL;
  const StencilCoefficients jacobi = {centreWeight, neighbourWeight};

  const CoverageProblem problem = coverageProblem(stations);
  field.upload(problem.initial, problem.fixed);
  const Convergence converged = sweepUntilConverged(field, jacobi);

  const Field expected = coverageOnCpu(problem, jacobi, converged.sweeps);
  const double error = maxRelativeDifference(expected, converged.coverage);
  const bool passed = error < MAX_RELATIVE_ERROR;

  std::printf("\n== Part A: station coverage (decay length %.0f m, lambda = %.5f)\n",
              DECAY_LENGTH_METRES,
              lambda);
  std::printf(
      "   converged after %d sweeps (last change %.1e) in %.1f ms: %s (max rel err vs CPU %.1e)\n",
      converged.sweeps,
      converged.lastChange,
      converged.ms,
      passed ? "PASS" : "FAIL",
      error);
  std::printf("   coverage map (S = docking station, north at the top):\n");
  printAsciiMap(converged.coverage, COVERAGE_MAP_SHADE_MAX, stations, StationMarks::Show);

  const ZoneMeans means = zoneMeans(converged.coverage);
  printZoneMeans(means);
  printWorstZones(means);
  return passed;
}

struct DemandSpot {
  Position centre;
  double spreadMetres;
};

const DemandSpot UNIVERSITY = {{900, 2500}, 220};
const DemandSpot OFFICES = {{3100, 3000}, 200};

using RequestsPerMinute = std::vector<std::vector<int>>;

static void printClock(int minuteOfDay, int second) {
  std::printf(
      "%02d:%02d:%02d", minuteOfDay / MINUTES_PER_HOUR, minuteOfDay % MINUTES_PER_HOUR, second);
}

static void printRequest(int minute, int requestOfMinute, int requestId, Position origin) {
  std::printf("   request %d at ", requestId);
  printClock(FIRST_SIMULATED_MINUTE_OF_DAY + minute,
             requestOfMinute * SECONDS_BETWEEN_REQUESTS_OF_A_MINUTE);
  std::printf(" from (%d m, %d m)\n", origin.x, origin.y);
}

static Position nearSpot(SplitMix64& rng, const DemandSpot& spot) {
  Position position;
  position.x = spot.centre.x + static_cast<int>(rng.standardNormal() * spot.spreadMetres);
  position.y = spot.centre.y + static_cast<int>(rng.standardNormal() * spot.spreadMetres);
  return position;
}

static RequestsPerMinute makeEveningRequests(int& requestCount) {
  SplitMix64 rng(DEMAND_SEED);
  RequestsPerMinute cells(SIMULATED_MINUTES);
  requestCount = 0;

  for (int minute = 0; minute < SIMULATED_MINUTES; ++minute) {
    const int requestsThisMinute = MIN_REQUESTS_PER_MINUTE + rng.below(EXTRA_REQUESTS_RANGE);
    const double shareStillAtUniversity = 1.0 - minute / (double)SIMULATED_MINUTES;

    for (int q = 0; q < requestsThisMinute; ++q) {
      const DemandSpot& spot = rng.uniform() < shareStillAtUniversity ? UNIVERSITY : OFFICES;
      const Position origin = nearSpot(rng, spot);
      if (!insideMap(origin)) {
        continue;
      }
      if (onMapEdge(cellRow(origin.y), cellCol(origin.x))) {
        continue;
      }

      cells[minute].push_back(cellOf(origin));
      if (requestCount < REQUESTS_TO_SHOW) {
        printRequest(minute, q, requestCount, origin);
      }
      ++requestCount;
    }
  }
  return cells;
}

static int diffusionStepsPerMinute() {
  const int refinement = GRID_SIDE / BASE_GRID_SIDE;
  return STEPS_PER_MINUTE_ON_BASE_GRID * refinement * refinement;
}

static StencilCoefficients diffusionCoefficients(int stepsPerMinute) {
  const float keptPerStep = (float)std::pow(DEMAND_KEPT_PER_MINUTE, 1.0 / stepsPerMinute);
  const float centreWeight = keptPerStep * (1.0f - NEIGHBOURS_PER_CELL * DIFFUSION_ALPHA);
  const float neighbourWeight = keptPerStep * DIFFUSION_ALPHA;
  return {centreWeight, neighbourWeight};
}

static Zone hottestZone(const ZoneMeans& means) {
  Zone hottest = {0, 0};
  for (int i = 0; i < ZONES_PER_SIDE; ++i) {
    for (int j = 0; j < ZONES_PER_SIDE; ++j) {
      const Zone zone = {i, j};
      if (means.at(zone) > means.at(hottest)) {
        hottest = zone;
      }
    }
  }
  return hottest;
}

static Position cellCentre(int cell) {
  const double size = cellSizeMetres();
  Position centre;
  centre.x = static_cast<int>((cell % GRID_SIDE + 0.5) * size);
  centre.y = static_cast<int>(MAP_SIDE_METRES - (cell / GRID_SIDE + 0.5) * size);
  return centre;
}

static void printSnapshot(int minutesElapsed, const Field& demand) {
  const int hottestCell = (int)(std::max_element(demand.begin(), demand.end()) - demand.begin());
  const Position centre = cellCentre(hottestCell);
  const Zone zone = hottestZone(zoneMeans(demand));

  std::printf("   ");
  printClock(FIRST_SIMULATED_MINUTE_OF_DAY + minutesElapsed, 0);
  std::printf("  hottest zone %s, hottest cell around (%d m, %d m)\n",
              nameOf(zone).c_str(),
              centre.x,
              centre.y);
}

class RequestInjector {
 public:
  RequestInjector() {
    CHECK(cudaMalloc(&cells_, INJECT_BLOCK_DIM * sizeof(int)));
  }

  ~RequestInjector() {
    CHECK(cudaFree(cells_));
  }

  void inject(float* demand, const std::vector<int>& cells) {
    const int n = (int)cells.size();
    if (n == 0) {
      return;
    }
    CHECK(cudaMemcpy(cells_, cells.data(), n * sizeof(int), cudaMemcpyHostToDevice));
    launch(inject_kernel, dim3(1), dim3(INJECT_BLOCK_DIM), demand, (const int*)cells_, n);
  }

 private:
  int* cells_;
};

static bool runDemandDiffusion(DeviceField& field, const std::vector<Station>& stations) {
  const int stepsPerMinute = diffusionStepsPerMinute();
  const StencilCoefficients diffusion = diffusionCoefficients(stepsPerMinute);
  const FixedMask fixed = mapEdgeMask();
  const Field empty(cellCount(), 0.0f);
  field.upload(empty, fixed);

  std::printf("\n== Part B: rental demand diffusion from ");
  printClock(FIRST_SIMULATED_MINUTE_OF_DAY, 0);
  std::printf(" for %d minutes, first requests:\n", SIMULATED_MINUTES);
  int requestCount = 0;
  const RequestsPerMinute requests = makeEveningRequests(requestCount);
  std::printf("   %d ride requests, %d diffusion steps per simulated minute\n",
              requestCount,
              stepsPerMinute);

  RequestInjector injector;
  Field latest = empty;
  Field older = empty;
  Field snapshot(cellCount());
  Timer timer;
  for (int minute = 0; minute < SIMULATED_MINUTES; ++minute) {
    injector.inject(field.latest(), requests[minute]);
    for (int cell : requests[minute]) {
      latest[cell] += 1.0f;
    }

    for (int s = 0; s < stepsPerMinute; ++s) {
      field.sweep(diffusion);
      sweepOnCpu(latest, older, fixed, diffusion);
    }

    const int minutesElapsed = minute + 1;
    if (minutesElapsed % SNAPSHOT_EVERY_MINUTES == 0) {
      snapshot = field.downloadLatest();
      printSnapshot(minutesElapsed, snapshot);
    }
  }
  CHECK(cudaGetLastError());
  const double ms = timer.ms();

  const double error = maxRelativeDifference(latest, snapshot);
  const bool passed = error < MAX_RELATIVE_ERROR;
  std::printf(
      "   %d sweeps + %d injections, %.1f ms (GPU and CPU reference together): %s (max rel err "
      "%.1e)\n",
      SIMULATED_MINUTES * stepsPerMinute,
      SIMULATED_MINUTES,
      ms,
      passed ? "PASS" : "FAIL",
      error);
  std::printf("   demand at ");
  printClock(FIRST_SIMULATED_MINUTE_OF_DAY + SIMULATED_MINUTES, 0);
  std::printf(" (darker = more recent requests nearby):\n");
  printAsciiMap(
      snapshot, *std::max_element(snapshot.begin(), snapshot.end()), stations, StationMarks::Hide);
  return passed;
}

static bool runBothParts(const std::vector<Station>& stations) {
  DeviceField field;
  bool allPassed = true;
  allPassed = runStationCoverage(field, stations) && allPassed;
  allPassed = runDemandDiffusion(field, stations) && allPassed;
  return allPassed;
}

int main() {
  std::printf(
      "Bike-share station coverage and demand diffusion, %d x %d cells (%.0f m), "
      "backend: %s\n",
      GRID_SIDE,
      GRID_SIDE,
      cellSizeMetres(),
      CUDA_BACKEND);

  const std::vector<Station> stations = makeStations();
  const bool allPassed = runBothParts(stations);

  std::printf("\n%s\n", allPassed ? "ALL PASS" : "SOME TESTS FAILED");
  return allPassed ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. bikeshare_station_coverage.cu -o bikeshare_station_coverage && ./bikeshare_station_coverage

## `stencil.cu`

In [ ]:
%%writefile stencil.cu
#include "cuda_compat.h"

#include <cmath>
#include <cstdio>
#include <random>
#include <utility>
#include <vector>

constexpr int STENCIL_POINTS = 7;
constexpr float COEFFICIENTS[STENCIL_POINTS] = {0.40f, 0.11f, 0.09f, 0.12f, 0.08f, 0.13f, 0.07f};

constexpr bool allDifferent(const float (&values)[STENCIL_POINTS]) {
  for (int a = 0; a < STENCIL_POINTS; ++a) {
    for (int b = a + 1; b < STENCIL_POINTS; ++b) {
      if (values[a] == values[b]) {
        return false;
      }
    }
  }
  return true;
}

constexpr bool addUpToOne(const float (&values)[STENCIL_POINTS]) {
  float sum = 0.0f;
  for (float value : values) {
    sum += value;
  }
  return sum > 0.9999f && sum < 1.0001f;
}

static_assert(allDifferent(COEFFICIENTS),
              "every neighbour has its own coefficient, so a kernel that mixes up two "
              "neighbours fails the check");
static_assert(addUpToOne(COEFFICIENTS),
              "the coefficients form a weighted average, so repeated sweeps stay bounded");

__constant__ float c_c[STENCIL_POINTS];

#define IN_TILE_3D 8
#define OUT_TILE_3D (IN_TILE_3D - 2)

#define IN_TILE_2D 32
#define OUT_TILE_2D (IN_TILE_2D - 2)

__global__ void stencil_basic_kernel(const float* in, float* out, int N) {
  int i = (int)(blockIdx.z * blockDim.z + threadIdx.z);
  int j = (int)(blockIdx.y * blockDim.y + threadIdx.y);
  int k = (int)(blockIdx.x * blockDim.x + threadIdx.x);

  if (i >= 1 && i < N - 1 && j >= 1 && j < N - 1 && k >= 1 && k < N - 1) {
    out[i * N * N + j * N + k] =
        c_c[0] * in[i * N * N + j * N + k] + c_c[1] * in[i * N * N + j * N + (k - 1)] +
        c_c[2] * in[i * N * N + j * N + (k + 1)] + c_c[3] * in[i * N * N + (j - 1) * N + k] +
        c_c[4] * in[i * N * N + (j + 1) * N + k] + c_c[5] * in[(i - 1) * N * N + j * N + k] +
        c_c[6] * in[(i + 1) * N * N + j * N + k];
  }
}

__global__ void stencil_tiled_kernel(const float* in, float* out, int N) {
  int i = (int)(blockIdx.z * OUT_TILE_3D) + (int)threadIdx.z - 1;
  int j = (int)(blockIdx.y * OUT_TILE_3D) + (int)threadIdx.y - 1;
  int k = (int)(blockIdx.x * OUT_TILE_3D) + (int)threadIdx.x - 1;

  __shared__ float in_s[IN_TILE_3D][IN_TILE_3D][IN_TILE_3D];
  if (i >= 0 && i < N && j >= 0 && j < N && k >= 0 && k < N) {
    in_s[threadIdx.z][threadIdx.y][threadIdx.x] = in[i * N * N + j * N + k];
  }
  __syncthreads();

  if (i >= 1 && i < N - 1 && j >= 1 && j < N - 1 && k >= 1 && k < N - 1) {
    int tz = (int)threadIdx.z;
    int ty = (int)threadIdx.y;
    int tx = (int)threadIdx.x;
    if (tz >= 1 && tz < IN_TILE_3D - 1 && ty >= 1 && ty < IN_TILE_3D - 1 && tx >= 1 &&
        tx < IN_TILE_3D - 1) {
      out[i * N * N + j * N + k] = c_c[0] * in_s[tz][ty][tx] + c_c[1] * in_s[tz][ty][tx - 1] +
                                   c_c[2] * in_s[tz][ty][tx + 1] + c_c[3] * in_s[tz][ty - 1][tx] +
                                   c_c[4] * in_s[tz][ty + 1][tx] + c_c[5] * in_s[tz - 1][ty][tx] +
                                   c_c[6] * in_s[tz + 1][ty][tx];
    }
  }
}

__global__ void stencil_coarsened_kernel(const float* in, float* out, int N) {
  int iStart = (int)(blockIdx.z * OUT_TILE_2D);
  int j = (int)(blockIdx.y * OUT_TILE_2D) + (int)threadIdx.y - 1;
  int k = (int)(blockIdx.x * OUT_TILE_2D) + (int)threadIdx.x - 1;
  int ty = (int)threadIdx.y;
  int tx = (int)threadIdx.x;

  __shared__ float inPrev_s[IN_TILE_2D][IN_TILE_2D];
  __shared__ float inCurr_s[IN_TILE_2D][IN_TILE_2D];
  __shared__ float inNext_s[IN_TILE_2D][IN_TILE_2D];

  bool inPlane = j >= 0 && j < N && k >= 0 && k < N;
  if (inPlane && iStart - 1 >= 0 && iStart - 1 < N) {
    inPrev_s[ty][tx] = in[(iStart - 1) * N * N + j * N + k];
  }
  if (inPlane && iStart < N) {
    inCurr_s[ty][tx] = in[iStart * N * N + j * N + k];
  }

  for (int i = iStart; i < iStart + OUT_TILE_2D; ++i) {
    if (inPlane && i + 1 < N) {
      inNext_s[ty][tx] = in[(i + 1) * N * N + j * N + k];
    }
    __syncthreads();

    if (i >= 1 && i < N - 1 && j >= 1 && j < N - 1 && k >= 1 && k < N - 1) {
      if (ty >= 1 && ty < IN_TILE_2D - 1 && tx >= 1 && tx < IN_TILE_2D - 1) {
        out[i * N * N + j * N + k] = c_c[0] * inCurr_s[ty][tx] + c_c[1] * inCurr_s[ty][tx - 1] +
                                     c_c[2] * inCurr_s[ty][tx + 1] + c_c[3] * inCurr_s[ty - 1][tx] +
                                     c_c[4] * inCurr_s[ty + 1][tx] + c_c[5] * inPrev_s[ty][tx] +
                                     c_c[6] * inNext_s[ty][tx];
      }
    }
    __syncthreads();

    inPrev_s[ty][tx] = inCurr_s[ty][tx];
    inCurr_s[ty][tx] = inNext_s[ty][tx];
  }
}

__global__ void stencil_register_tiled_kernel(const float* in, float* out, int N) {
  int iStart = (int)(blockIdx.z * OUT_TILE_2D);
  int j = (int)(blockIdx.y * OUT_TILE_2D) + (int)threadIdx.y - 1;
  int k = (int)(blockIdx.x * OUT_TILE_2D) + (int)threadIdx.x - 1;
  int ty = (int)threadIdx.y;
  int tx = (int)threadIdx.x;

  float inPrev = 0.0f;
  float inCurr = 0.0f;
  float inNext = 0.0f;
  __shared__ float inCurr_s[IN_TILE_2D][IN_TILE_2D];

  bool inPlane = j >= 0 && j < N && k >= 0 && k < N;
  if (inPlane && iStart - 1 >= 0 && iStart - 1 < N) {
    inPrev = in[(iStart - 1) * N * N + j * N + k];
  }
  if (inPlane && iStart < N) {
    inCurr = in[iStart * N * N + j * N + k];
    inCurr_s[ty][tx] = inCurr;
  }

  for (int i = iStart; i < iStart + OUT_TILE_2D; ++i) {
    if (inPlane && i + 1 < N) {
      inNext = in[(i + 1) * N * N + j * N + k];
    }
    __syncthreads();

    if (i >= 1 && i < N - 1 && j >= 1 && j < N - 1 && k >= 1 && k < N - 1) {
      if (ty >= 1 && ty < IN_TILE_2D - 1 && tx >= 1 && tx < IN_TILE_2D - 1) {
        out[i * N * N + j * N + k] = c_c[0] * inCurr + c_c[1] * inCurr_s[ty][tx - 1] +
                                     c_c[2] * inCurr_s[ty][tx + 1] + c_c[3] * inCurr_s[ty - 1][tx] +
                                     c_c[4] * inCurr_s[ty + 1][tx] + c_c[5] * inPrev +
                                     c_c[6] * inNext;
      }
    }
    __syncthreads();

    inPrev = inCurr;
    inCurr = inNext;
    inCurr_s[ty][tx] = inNext;
  }
}

using Grid = std::vector<float>;

static void stencil_cpu(const Grid& in, Grid& out, int N) {
  const float* c = COEFFICIENTS;
  for (int i = 1; i < N - 1; ++i) {
    for (int j = 1; j < N - 1; ++j) {
      for (int k = 1; k < N - 1; ++k) {
        size_t p = (size_t)i * N * N + (size_t)j * N + k;
        out[p] = c[0] * in[p] + c[1] * in[p - 1] + c[2] * in[p + 1] + c[3] * in[p - N] +
                 c[4] * in[p + N] + c[5] * in[p - (size_t)N * N] + c[6] * in[p + (size_t)N * N];
      }
    }
  }
}

const int GRID_SIDE = BY_BACKEND(256, 40);
const int TIMED_LAUNCHES_AFTER_CHECK = BY_BACKEND(20, 0);
const int ITERATIVE_SWEEPS = BY_BACKEND(100, 3);
const unsigned RANDOM_SEED = 7;
const double MAX_RELATIVE_ERROR = 1e-5;

const int BASIC_BLOCK_DIM = 8;
const int BYTES_PER_FLOAT = sizeof(float);
const int STENCIL_MULTIPLICATIONS = 7;
const int STENCIL_ADDITIONS = 6;
const double FLOP_PER_POINT = STENCIL_MULTIPLICATIONS + STENCIL_ADDITIONS;
const int MARCHED_OUTPUT_PLANES = OUT_TILE_2D;
const int MARCHED_INPUT_PLANES = OUT_TILE_2D + 2;
const int COARSENED_SHARED_PLANES = 3;
const int REGISTER_TILED_SHARED_PLANES = 1;

struct DeviceGrids {
  float* in;
  float* out;
  size_t bytes;
};

struct LaunchShape {
  dim3 grid;
  dim3 block;
};

struct LaunchShapes {
  LaunchShape basic;
  LaunchShape tiled3D;
  LaunchShape marching2D;
};

static size_t pointCount() {
  return (size_t)GRID_SIDE * GRID_SIDE * GRID_SIDE;
}

static Grid randomGrid(unsigned seed) {
  std::mt19937 rng(seed);
  std::uniform_real_distribution<float> value(0.0f, 1.0f);
  Grid grid(pointCount());
  for (float& point : grid) {
    point = value(rng);
  }
  return grid;
}

static Grid sweepOnCpu(const Grid& input, int sweeps) {
  Grid current = input;
  Grid next = input;
  for (int s = 0; s < sweeps; ++s) {
    stencil_cpu(current, next, GRID_SIDE);
    std::swap(current, next);
  }
  return current;
}

struct Reference {
  Grid output;
  double ms;
};

static Reference oneSweepReference(const Grid& input) {
  Reference reference;
  reference.output = input;

  Timer timer;
  stencil_cpu(input, reference.output, GRID_SIDE);
  reference.ms = timer.ms();
  return reference;
}

static double maxRelativeError(const Grid& expected, const Grid& actual) {
  double worst = 0.0;
  for (size_t i = 0; i < expected.size(); ++i) {
    double difference = std::fabs((double)expected[i] - actual[i]);
    worst = std::fmax(worst, difference / (1.0 + std::fabs((double)expected[i])));
  }
  return worst;
}

static LaunchShape cubeOfBlocks(int blockDim, int outputsPerBlock) {
  const unsigned blocksPerSide = cdiv(GRID_SIDE, outputsPerBlock);
  return {dim3(blocksPerSide, blocksPerSide, blocksPerSide), dim3(blockDim, blockDim, blockDim)};
}

static LaunchShapes launchShapes() {
  LaunchShapes shapes;
  shapes.basic = cubeOfBlocks(BASIC_BLOCK_DIM, BASIC_BLOCK_DIM);
  shapes.tiled3D = cubeOfBlocks(IN_TILE_3D, OUT_TILE_3D);

  const unsigned blocksPerSide = cdiv(GRID_SIDE, OUT_TILE_2D);
  shapes.marching2D.grid = dim3(blocksPerSide, blocksPerSide, blocksPerSide);
  shapes.marching2D.block = dim3(IN_TILE_2D, IN_TILE_2D, 1);
  return shapes;
}

static DeviceGrids uploadToDevice(const Grid& input) {
  DeviceGrids device;
  device.bytes = input.size() * sizeof(float);
  CHECK(cudaMemcpyToSymbol(c_c, COEFFICIENTS, sizeof(COEFFICIENTS)));
  CHECK(cudaMalloc(&device.in, device.bytes));
  CHECK(cudaMalloc(&device.out, device.bytes));
  CHECK(cudaMemcpy(device.in, input.data(), device.bytes, cudaMemcpyHostToDevice));
  return device;
}

static void freeDevice(DeviceGrids& device) {
  CHECK(cudaFree(device.in));
  CHECK(cudaFree(device.out));
}

class KernelTable {
 public:
  KernelTable(const DeviceGrids& device, const Grid& expected)
      : device_(device), expected_(expected), actual_(expected.size()) {
    const double interiorSide = GRID_SIDE - 2;
    flop_ = FLOP_PER_POINT * interiorSide * interiorSide * interiorSide;
    bytesToReadAndWriteOnce_ = 2.0 * device.bytes;
  }

  void printHeader(double cpuMs) const {
    std::printf(
        "\n%-32s %10s %10s %10s  %s\n", "kernel (one sweep)", "ms", "GFLOP/s", "GB/s*", "check");
    std::printf("%-32s %10.3f %10.2f %10s  %s\n",
                "CPU reference (1 thread)",
                cpuMs,
                gflopPerSecond(cpuMs),
                "-",
                "-");
  }

  template <class LaunchKernel>
  void run(const char* name, LaunchKernel launchKernel) {
    startOutputAsCopyOfInput();
    Timer timer;
    launchKernel();
    CHECK(cudaGetLastError());
    double ms = timer.ms();

    CHECK(cudaMemcpy(actual_.data(), device_.out, device_.bytes, cudaMemcpyDeviceToHost));
    const double error = maxRelativeError(expected_, actual_);
    const bool passed = error < MAX_RELATIVE_ERROR;
    allPassed_ = allPassed_ && passed;

    if (TIMED_LAUNCHES_AFTER_CHECK > 0) {
      ms = averageMsOfRepeatedLaunches(launchKernel);
    }
    printRow(name, ms, passed, error);
  }

  bool allPassed() const {
    return allPassed_;
  }

 private:
  void startOutputAsCopyOfInput() {
    CHECK(cudaMemcpy(device_.out, device_.in, device_.bytes, cudaMemcpyDeviceToDevice));
  }

  template <class LaunchKernel>
  static double averageMsOfRepeatedLaunches(LaunchKernel launchKernel) {
    Timer timer;
    for (int r = 0; r < TIMED_LAUNCHES_AFTER_CHECK; ++r) {
      launchKernel();
    }
    return timer.ms() / TIMED_LAUNCHES_AFTER_CHECK;
  }

  double gflopPerSecond(double ms) const {
    return flop_ / (ms * 1e6);
  }

  double gigabytesPerSecond(double ms) const {
    return bytesToReadAndWriteOnce_ / (ms * 1e6);
  }

  void printRow(const char* name, double ms, bool passed, double error) const {
    std::printf("%-32s %10.3f %10.2f %10.2f  %s (max rel err %.1e)\n",
                name,
                ms,
                gflopPerSecond(ms),
                gigabytesPerSecond(ms),
                passed ? "PASS" : "FAIL",
                error);
  }

  const DeviceGrids& device_;
  const Grid& expected_;
  Grid actual_;
  double flop_;
  double bytesToReadAndWriteOnce_;
  bool allPassed_ = true;
};

static void runEveryKernel(KernelTable& table,
                           const DeviceGrids& device,
                           const LaunchShapes& shapes) {
  const float* in = device.in;
  float* out = device.out;
  const int N = GRID_SIDE;

  table.run("(a) basic", [&] {
    launch(stencil_basic_kernel, shapes.basic.grid, shapes.basic.block, in, out, N);
  });
  table.run("(b) shared-memory tiles 8^3", [&] {
    launch(stencil_tiled_kernel, shapes.tiled3D.grid, shapes.tiled3D.block, in, out, N);
  });
  table.run("(c) coarsened in z, 32^2", [&] {
    launch(stencil_coarsened_kernel, shapes.marching2D.grid, shapes.marching2D.block, in, out, N);
  });
  table.run("(d) register tiling, 32^2", [&] {
    launch(
        stencil_register_tiled_kernel, shapes.marching2D.grid, shapes.marching2D.block, in, out, N);
  });

  std::printf("* GB/s counts only the minimum traffic (read the grid once, write it once).\n");
  std::printf("  In CPU emulation the times measure thread scheduling, not memory behaviour.\n");
}

static bool runIterativeSweeps(const Grid& input,
                               const DeviceGrids& device,
                               const LaunchShapes& shapes) {
  float* src = device.in;
  float* dst = device.out;
  CHECK(cudaMemcpy(src, input.data(), device.bytes, cudaMemcpyHostToDevice));
  CHECK(cudaMemcpy(dst, src, device.bytes, cudaMemcpyDeviceToDevice));

  Timer timer;
  for (int s = 0; s < ITERATIVE_SWEEPS; ++s) {
    launch(stencil_register_tiled_kernel,
           shapes.marching2D.grid,
           shapes.marching2D.block,
           (const float*)src,
           dst,
           GRID_SIDE);
    std::swap(src, dst);
  }
  CHECK(cudaGetLastError());
  const double ms = timer.ms();

  Grid result(input.size());
  CHECK(cudaMemcpy(result.data(), src, device.bytes, cudaMemcpyDeviceToHost));
  const Grid expected = sweepOnCpu(input, ITERATIVE_SWEEPS);
  const double error = maxRelativeError(expected, result);
  const bool passed = error < MAX_RELATIVE_ERROR;

  std::printf(
      "\n%d sweeps of (d) with double buffering: %.3f ms (%.3f ms/sweep)  %s (max rel err %.1e)\n",
      ITERATIVE_SWEEPS,
      ms,
      ms / ITERATIVE_SWEEPS,
      passed ? "PASS" : "FAIL",
      error);
  return passed;
}

static double cube(int side) {
  return (double)side * side * side;
}

static double square(int side) {
  return (double)side * side;
}

static void printResourceRow(
    const char* name, int threads, int sharedBytes, double outputs, double loads) {
  const double intensity = FLOP_PER_POINT * outputs / (BYTES_PER_FLOAT * (loads + outputs));
  std::printf("  %-34s %8d %10d %12.0f %10.2f\n", name, threads, sharedBytes, outputs, intensity);
}

static void printResourceTable() {
  std::printf(
      "\nPer-block resources and arithmetic intensity (13 FLOP per output, 4 B per load and per "
      "store):\n");
  std::printf("  %-34s %8s %10s %12s %10s\n", "kernel", "threads", "shared B", "outputs", "FLOP/B");

  const int basicThreads = BASIC_BLOCK_DIM * BASIC_BLOCK_DIM * BASIC_BLOCK_DIM;
  printResourceRow("(a) basic (7 loads per output)", basicThreads, 0, 1, STENCIL_POINTS);

  for (int T : {8, 10}) {
    char name[64];
    std::snprintf(name, sizeof name, "(b) 3D tile T=%d", T);
    printResourceRow(name, (int)cube(T), BYTES_PER_FLOAT * (int)cube(T), cube(T - 2), cube(T));
  }

  const int T = IN_TILE_2D;
  const int planeThreads = T * T;
  const int planeBytes = BYTES_PER_FLOAT * T * T;
  const double outputsPerPlane = square(T - 2);
  const double loadsPerPlane = square(T);
  printResourceRow("(c) coarsened T=32, 30 planes",
                   planeThreads,
                   COARSENED_SHARED_PLANES * planeBytes,
                   MARCHED_OUTPUT_PLANES * outputsPerPlane,
                   MARCHED_INPUT_PLANES * loadsPerPlane);
  printResourceRow("(c) same, one plane (steady state)",
                   planeThreads,
                   COARSENED_SHARED_PLANES * planeBytes,
                   outputsPerPlane,
                   loadsPerPlane);
  printResourceRow("(d) register tiled T=32, 30 planes",
                   planeThreads,
                   REGISTER_TILED_SHARED_PLANES * planeBytes,
                   MARCHED_OUTPUT_PLANES * outputsPerPlane,
                   MARCHED_INPUT_PLANES * loadsPerPlane);
  std::printf("  limit for very large tiles: 13/8 = 1.625 FLOP/B (H100 ridge: 20.0)\n");
}

int main() {
  const Grid input = randomGrid(RANDOM_SEED);
  const double megabytesPerArray = input.size() * sizeof(float) / 1e6;
  std::printf("3D seven-point stencil, %d^3 grid (%.1f MB per array), backend: %s\n",
              GRID_SIDE,
              megabytesPerArray,
              CUDA_BACKEND);

  const Reference reference = oneSweepReference(input);

  DeviceGrids device = uploadToDevice(input);
  const LaunchShapes shapes = launchShapes();

  KernelTable table(device, reference.output);
  table.printHeader(reference.ms);
  runEveryKernel(table, device, shapes);

  const bool iterativePassed = runIterativeSweeps(input, device, shapes);
  printResourceTable();

  freeDevice(device);
  const bool allPassed = table.allPassed() && iterativePassed;
  std::printf("\n%s\n", allPassed ? "ALL PASS" : "SOME KERNEL FAILED");
  return allPassed ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. stencil.cu -o stencil && ./stencil

## `stencil_basics.cu`

In [ ]:
%%writefile stencil_basics.cu
#include "cuda_compat.h"

#include <cmath>
#include <cstdio>
#include <utility>
#include <vector>

__global__ void finite_difference_kernel(const double* F, double* FD, int n, double h) {
  int i = (int)(blockIdx.x * blockDim.x + threadIdx.x);
  if (i >= 1 && i < n - 1) {
    FD[i] = (-1.0 / (2.0 * h)) * F[i - 1] + (1.0 / (2.0 * h)) * F[i + 1];
  }
}

const int CHAPTER_GRID_POINTS = 7;
const int FINEST_GRID_POINTS = CHAPTER_GRID_POINTS * 512;
const int FINITE_DIFFERENCE_BLOCK_DIM = 128;
const double MIN_ERROR_RATIO_FOR_SECOND_ORDER = 3.5;
const double MAX_ERROR_RATIO_FOR_SECOND_ORDER = 4.5;

using Samples = std::vector<double>;

static int pointsWithHalfTheSpacing(int points) {
  return 2 * points - 1;
}

static Samples sampleSine(int n, double h) {
  Samples F(n);
  for (int i = 0; i < n; ++i) {
    F[i] = std::sin(i * h);
  }
  return F;
}

static Samples finiteDifferenceOnDevice(const Samples& F, double h) {
  const int n = (int)F.size();
  const size_t bytes = n * sizeof(double);
  Samples FD(n, 0.0);

  double* dF;
  double* dFD;
  CHECK(cudaMalloc(&dF, bytes));
  CHECK(cudaMalloc(&dFD, bytes));
  CHECK(cudaMemcpy(dF, F.data(), bytes, cudaMemcpyHostToDevice));
  CHECK(cudaMemset(dFD, 0, bytes));

  launch(finite_difference_kernel,
         dim3(cdiv(n, FINITE_DIFFERENCE_BLOCK_DIM)),
         dim3(FINITE_DIFFERENCE_BLOCK_DIM),
         static_cast<const double*>(dF),
         dFD,
         n,
         h);
  CHECK(cudaGetLastError());

  CHECK(cudaMemcpy(FD.data(), dFD, bytes, cudaMemcpyDeviceToHost));
  CHECK(cudaFree(dF));
  CHECK(cudaFree(dFD));
  return FD;
}

static void printChapterGrid(const Samples& F, const Samples& FD, double h) {
  const int n = (int)F.size();

  std::printf("   F  = ");
  for (int i = 0; i < n; ++i) {
    std::printf("%5.2f ", F[i]);
  }

  std::printf("\n   FD = %5s ", "-");
  for (int i = 1; i < n - 1; ++i) {
    std::printf("%5.2f ", FD[i]);
  }
  std::printf("%5s   (interior points only)\n", "-");

  std::printf("   cos= %5s ", "-");
  for (int i = 1; i < n - 1; ++i) {
    std::printf("%5.2f ", std::cos(i * h));
  }
  std::printf("%5s\n", "-");
}

static double maxErrorAgainstCosine(const Samples& FD, double h) {
  const int n = (int)FD.size();
  double error = 0.0;
  for (int i = 1; i < n - 1; ++i) {
    error = std::fmax(error, std::fabs(FD[i] - std::cos(i * h)));
  }
  return error;
}

static bool isSecondOrderRatio(double ratio) {
  return ratio > MIN_ERROR_RATIO_FOR_SECOND_ORDER && ratio < MAX_ERROR_RATIO_FOR_SECOND_ORDER;
}

static bool runFiniteDifference() {
  const double PI = std::acos(-1.0);
  std::printf("Part 1: derivative of sin(x) on [0, pi] with the 3-point stencil\n");
  std::printf("%8s %12s %16s %12s\n", "points", "h", "max |FD - cos|", "error ratio");

  double previousError = 0.0;
  bool passed = true;
  for (int n = CHAPTER_GRID_POINTS; n <= FINEST_GRID_POINTS; n = pointsWithHalfTheSpacing(n)) {
    const double h = PI / (n - 1);
    const Samples F = sampleSine(n, h);
    const Samples FD = finiteDifferenceOnDevice(F, h);

    if (n == CHAPTER_GRID_POINTS) {
      printChapterGrid(F, FD, h);
    }

    const double error = maxErrorAgainstCosine(FD, h);
    const bool hasPreviousRow = previousError > 0.0;
    if (hasPreviousRow) {
      const double ratio = previousError / error;
      std::printf("%8d %12.6f %16.3e %12.2f\n", n, h, error, ratio);
      passed = passed && isSecondOrderRatio(ratio);
    } else {
      std::printf("%8d %12.6f %16.3e %12s\n", n, h, error, "-");
    }
    previousError = error;
  }

  std::printf("Error ratio ~4 each time h halves: the error is O(h^2).  %s\n\n",
              passed ? "PASS" : "FAIL");
  return passed;
}

#define IN_TILE_DIM 32
#define OUT_TILE_DIM (IN_TILE_DIM - 2)

__global__ void stencil2d_basic(const float* in, float* out, int N) {
  int i = (int)(blockIdx.y * blockDim.y + threadIdx.y);
  int j = (int)(blockIdx.x * blockDim.x + threadIdx.x);

  if (i >= 1 && i < N - 1 && j >= 1 && j < N - 1) {
    out[i * N + j] =
        0.25f * (in[(i - 1) * N + j] + in[(i + 1) * N + j] + in[i * N + j - 1] + in[i * N + j + 1]);
  }
}

__global__ void stencil2d_tiled(const float* in, float* out, int N) {
  int i = (int)(blockIdx.y * OUT_TILE_DIM + threadIdx.y) - 1;
  int j = (int)(blockIdx.x * OUT_TILE_DIM + threadIdx.x) - 1;

  __shared__ float in_s[IN_TILE_DIM][IN_TILE_DIM];
  if (i >= 0 && i < N && j >= 0 && j < N) {
    in_s[threadIdx.y][threadIdx.x] = in[i * N + j];
  }
  __syncthreads();

  int ty = (int)threadIdx.y;
  int tx = (int)threadIdx.x;
  bool interiorPoint = i >= 1 && i < N - 1 && j >= 1 && j < N - 1;
  bool innerThread = ty >= 1 && ty < IN_TILE_DIM - 1 && tx >= 1 && tx < IN_TILE_DIM - 1;
  if (interiorPoint && innerThread) {
    out[i * N + j] =
        0.25f * (in_s[ty - 1][tx] + in_s[ty + 1][tx] + in_s[ty][tx - 1] + in_s[ty][tx + 1]);
  }
}

static void stencil2d_cpu(const std::vector<float>& in, std::vector<float>& out, int N) {
  for (int i = 1; i < N - 1; ++i) {
    for (int j = 1; j < N - 1; ++j) {
      out[i * N + j] = 0.25f * (in[(i - 1) * N + j] + in[(i + 1) * N + j] + in[i * N + j - 1] +
                                in[i * N + j + 1]);
    }
  }
}

const int PLATE_SIDE = BY_BACKEND(512, 40);
const int SWEEPS = BY_BACKEND(20000, 150);
const float HOT_EDGE_TEMPERATURE = 100.0f;
const int BASIC_BLOCK_DIM = 16;
const double MAX_DIFFERENCE_FROM_CPU = 1e-3;

const int BYTES_PER_FLOAT = sizeof(float);
const int FIVE_POINT_MULTIPLICATIONS = 5;
const int FIVE_POINT_ADDITIONS = 4;
const double FIVE_POINT_FLOP = FIVE_POINT_MULTIPLICATIONS + FIVE_POINT_ADDITIONS;
const int FIVE_POINT_LOADS = 5;
const int SMALLEST_CONVOLUTION_DIM = 3;

const char* const TEMPERATURE_SHADES = " .:-=+*#%@";
const int DARKEST_SHADE = 9;
const int PICTURE_ROWS = 8;
const int PICTURE_COLUMNS = 32;

using Plate = std::vector<float>;

enum class PlateKernel { Basic, Tiled };

const PlateKernel PLATE_KERNELS[] = {PlateKernel::Basic, PlateKernel::Tiled};

static const char* nameOf(PlateKernel kernel) {
  return kernel == PlateKernel::Basic ? "basic" : "tiled";
}

static Plate plateWithHotTopEdge(int N) {
  Plate plate((size_t)N * N, 0.0f);
  for (int j = 0; j < N; ++j) {
    plate[j] = HOT_EDGE_TEMPERATURE;
  }
  return plate;
}

static Plate sweepOnCpu(const Plate& plate, int N, int sweeps) {
  Plate current = plate;
  Plate next = plate;
  for (int s = 0; s < sweeps; ++s) {
    stencil2d_cpu(current, next, N);
    std::swap(current, next);
  }
  return current;
}

static void launchSweep(PlateKernel kernel, const float* in, float* out, int N) {
  if (kernel == PlateKernel::Basic) {
    launch(stencil2d_basic,
           dim3(cdiv(N, BASIC_BLOCK_DIM), cdiv(N, BASIC_BLOCK_DIM)),
           dim3(BASIC_BLOCK_DIM, BASIC_BLOCK_DIM),
           in,
           out,
           N);
  } else {
    launch(stencil2d_tiled,
           dim3(cdiv(N, OUT_TILE_DIM), cdiv(N, OUT_TILE_DIM)),
           dim3(IN_TILE_DIM, IN_TILE_DIM),
           in,
           out,
           N);
  }
}

static double maxAbsoluteDifference(const Plate& a, const Plate& b) {
  double worst = 0.0;
  for (size_t p = 0; p < a.size(); ++p) {
    worst = std::fmax(worst, std::fabs(a[p] - b[p]));
  }
  return worst;
}

static bool checkKernelOnPlate(
    PlateKernel kernel, const Plate& plate, const Plate& expected, float* bufferA, float* bufferB) {
  const int N = PLATE_SIDE;
  const size_t bytes = plate.size() * sizeof(float);
  CHECK(cudaMemcpy(bufferA, plate.data(), bytes, cudaMemcpyHostToDevice));
  CHECK(cudaMemcpy(bufferB, plate.data(), bytes, cudaMemcpyHostToDevice));

  float* src = bufferA;
  float* dst = bufferB;
  Timer timer;
  for (int s = 0; s < SWEEPS; ++s) {
    launchSweep(kernel, src, dst, N);
    std::swap(src, dst);
  }
  const double ms = timer.ms();
  CHECK(cudaGetLastError());

  Plate result(plate.size());
  CHECK(cudaMemcpy(result.data(), src, bytes, cudaMemcpyDeviceToHost));
  const double difference = maxAbsoluteDifference(result, expected);
  const bool passed = difference < MAX_DIFFERENCE_FROM_CPU;

  std::printf("  %-6s %9.2f ms for %d sweeps, max diff vs CPU %.2e  %s\n",
              nameOf(kernel),
              ms,
              SWEEPS,
              difference,
              passed ? "PASS" : "FAIL");
  return passed;
}

static double idealIntensity(double flopPerPoint) {
  const double bytesPerPoint = 2.0 * BYTES_PER_FLOAT;
  return flopPerPoint / bytesPerPoint;
}

static void printPlateIntensities() {
  const double basicBytes = FIVE_POINT_LOADS * BYTES_PER_FLOAT + BYTES_PER_FLOAT;
  const double basic = FIVE_POINT_FLOP / basicBytes;

  const double inputTile = IN_TILE_DIM;
  const double outputTile = OUT_TILE_DIM;
  const double tileBytes =
      BYTES_PER_FLOAT * inputTile * inputTile + BYTES_PER_FLOAT * outputTile * outputTile;
  const double tiled = FIVE_POINT_FLOP * outputTile * outputTile / tileBytes;

  const double convolutionFlop = 2.0 * SMALLEST_CONVOLUTION_DIM * SMALLEST_CONVOLUTION_DIM;

  std::printf(
      "  intensity (general 5-point, 9 FLOP/point): basic %.3f, tiled %.3f, ideal %.3f FLOP/B\n",
      basic,
      tiled,
      idealIntensity(FIVE_POINT_FLOP));
  std::printf(
      "  (a 3x3 convolution: ideal %.2f FLOP/B; the 5-point stencil reuses each input far less)\n",
      idealIntensity(convolutionFlop));
}

static char shadeOf(float temperature) {
  const int shade = static_cast<int>(temperature / HOT_EDGE_TEMPERATURE * DARKEST_SHADE + 0.5f);
  if (shade < 0) {
    return TEMPERATURE_SHADES[0];
  }
  if (shade > DARKEST_SHADE) {
    return TEMPERATURE_SHADES[DARKEST_SHADE];
  }
  return TEMPERATURE_SHADES[shade];
}

static void printTemperaturePicture(const Plate& plate, int N) {
  std::printf("  temperature (top edge hot):\n");
  for (int r = 0; r < PICTURE_ROWS; ++r) {
    const size_t plateRow = static_cast<size_t>(r * (N - 1) / (PICTURE_ROWS - 1));
    std::printf("    ");
    for (int c = 0; c < PICTURE_COLUMNS; ++c) {
      const int plateCol = c * (N - 1) / (PICTURE_COLUMNS - 1);
      std::putchar(shadeOf(plate[plateRow * N + plateCol]));
    }
    std::putchar('\n');
  }
}

static bool runHeatPlate() {
  const int N = PLATE_SIDE;
  std::printf("Part 2: 2D 5-point stencil, %dx%d plate, top edge 100, other edges 0, %d sweeps\n",
              N,
              N,
              SWEEPS);

  const Plate plate = plateWithHotTopEdge(N);
  const Plate expected = sweepOnCpu(plate, N, SWEEPS);

  const size_t bytes = plate.size() * sizeof(float);
  float* bufferA;
  float* bufferB;
  CHECK(cudaMalloc(&bufferA, bytes));
  CHECK(cudaMalloc(&bufferB, bytes));

  bool passed = true;
  for (PlateKernel kernel : PLATE_KERNELS) {
    passed = checkKernelOnPlate(kernel, plate, expected, bufferA, bufferB) && passed;
  }

  CHECK(cudaFree(bufferA));
  CHECK(cudaFree(bufferB));

  printPlateIntensities();
  printTemperaturePicture(expected, N);
  return passed;
}

int main() {
  std::printf("Stencil basics (chapter 8), backend: %s\n\n", CUDA_BACKEND);

  bool passed = runFiniteDifference();
  passed = runHeatPlate() && passed;

  std::printf("\n%s\n", passed ? "ALL PASS" : "SOME FAILED");
  return passed ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. stencil_basics.cu -o stencil_basics && ./stencil_basics